# Extraemos Productos

In [2]:
import pandas as pd
import numpy as np
import requests
import pyarrow
import sqlalchemy
from dotenv import load_dotenv

In [3]:
import json
from pathlib import Path
from datetime import datetime

In [4]:
url = "https://dummyjson.com/products"
response = requests.get(url, timeout=30) 
print("Status code:", response.status_code)

Status code: 200


In [5]:
data = response.json()

In [6]:
print(type(data))

<class 'dict'>


In [7]:
data.keys()

dict_keys(['products', 'total', 'skip', 'limit'])

In [8]:
print("Total: ", data["total"])
print("Registros recibidos: ", len(data["products"]))
print("Limite: ", data["limit"])
print("Omitidos: ", data["skip"])

Total:  194
Registros recibidos:  30
Limite:  30
Omitidos:  0


#### Convertimos a dataframes

In [9]:
products_df = pd.DataFrame(data["products"])

In [10]:
products_df.head()

,id,title,description,category,price,discountPercentage,rating,stock,tags,brand,...,dimensions,warrantyInformation,shippingInformation,availabilityStatus,reviews,returnPolicy,minimumOrderQuantity,meta,images,thumbnail
0,1,Essence Mascara Lash Princess,The Essence Mascara Lash Princess is a popular...,beauty,9.99,10.48,2.56,99,"[beauty, mascara]",Essence,...,"{'width': 15.14, 'height': 13.08, 'depth': 22.99}",1 week warranty,Ships in 3-5 business days,In Stock,"[{'rating': 3, 'comment': 'Would not recommend...",No return policy,48,"{'createdAt': '2025-10-09T14:47:01.588Z', 'upd...",[https://cdn.dummyjson.com/product-images/beau...,https://cdn.dummyjson.com/product-images/beaut...
1,2,Eyeshadow Palette with Mirror,The Eyeshadow Palette with Mirror offers a ver...,beauty,19.99,18.19,2.86,34,"[beauty, eyeshadow]",Glamour Beauty,...,"{'width': 9.26, 'height': 22.47, 'depth': 27.67}",1 year warranty,Ships in 2 weeks,In Stock,"[{'rating': 5, 'comment': 'Great product!', 'd...",7 days return policy,20,"{'createdAt': '2024-12-03T11:29:45.706Z', 'upd...",[https://cdn.dummyjson.com/product-images/beau...,https://cdn.dummyjson.com/product-images/beaut...
2,3,Powder Canister,The Powder Canister is a finely milled setting...,beauty,14.99,9.84,4.64,89,"[beauty, face powder]",Velvet Touch,...,"{'width': 29.27, 'height': 27.93, 'depth': 20.59}",3 months warranty,Ships in 1-2 business days,In Stock,"[{'rating': 4, 'comment': 'Would buy again!', ...",No return policy,22,"{'createdAt': '2025-01-08T18:30:12.844Z', 'upd...",[https://cdn.dummyjson.com/product-images/beau...,https://cdn.dummyjson.com/product-images/beaut...
3,4,Red Lipstick,The Red Lipstick is a classic and bold choice ...,beauty,12.99,12.16,4.36,91,"[beauty, lipstick]",Chic Cosmetics,...,"{'width': 18.11, 'height': 28.38, 'depth': 22.17}",3 year warranty,Ships in 1 week,In Stock,"[{'rating': 4, 'comment': 'Great product!', 'd...",7 days return policy,40,"{'createdAt': '2025-11-22T23:28:00.911Z', 'upd...",[https://cdn.dummyjson.com/product-images/beau...,https://cdn.dummyjson.com/product-images/beaut...
4,5,Red Nail Polish,The Red Nail Polish offers a rich and glossy r...,beauty,8.99,11.44,4.32,79,"[beauty, nail polish]",Nail Couture,...,"{'width': 21.63, 'height': 16.48, 'depth': 29.84}",1 month warranty,Ships overnight,In Stock,"[{'rating': 2, 'comment': 'Poor quality!', 'da...",No return policy,22,"{'createdAt': '2025-10-03T07:39:53.470Z', 'upd...",[https://cdn.dummyjson.com/product-images/beau...,https://cdn.dummyjson.com/product-images/beaut...


In [11]:
products_df.info()

<class 'pandas.DataFrame'>
RangeIndex: 30 entries, 0 to 29
Data columns (total 22 columns):
 #   Column                Non-Null Count  Dtype  
---  ------                --------------  -----  
 0   id                    30 non-null     int64  
 1   title                 30 non-null     str    
 2   description           30 non-null     str    
 3   category              30 non-null     str    
 4   price                 30 non-null     float64
 5   discountPercentage    30 non-null     float64
 6   rating                30 non-null     float64
 7   stock                 30 non-null     int64  
 8   tags                  30 non-null     object 
 9   brand                 15 non-null     str    
 10  sku                   30 non-null     str    
 11  weight                30 non-null     int64  
 12  dimensions            30 non-null     object 
 13  warrantyInformation   30 non-null     str    
 14  shippingInformation   30 non-null     str    
 15  availabilityStatus    30 non-null   

#### Guardamos la info bronze

In [12]:
# Comprobamos que la ruta exista
bronze_path = Path("../data/bronze")
bronze_path.mkdir(parents=True, exist_ok=True)

In [13]:
# Guardamos la respuesta de la API

file_path = bronze_path / "products_raw.json"

with open(file_path, "w", encoding="utf-8") as file:
    json.dump(data, file, ensure_ascii=False, indent=4)

#### Debemos completar los 194 datos en total, apenas llevabamos 30

In [14]:
url = "https://dummyjson.com/products"

response = requests.get(
    url,
    params={"limit": 30, "skip": 30},
    timeout=30
)

response.raise_for_status()

page_2 = response.json()

print("Productos recibidos:", len(page_2["products"]))
print("Primer producto:", page_2["products"][0]["id"])

Productos recibidos: 30
Primer producto: 31


#### Se puede automatizar

In [15]:
all_products = []
skip = 0
limit = 30

while True:
    response = requests.get(
        url,
        params={"limit": limit, "skip": skip},
        timeout=30
    )

    response.raise_for_status()

    batch = response.json()["products"]

    all_products.extend(batch)

    skip += limit

    if len(all_products) >= data["total"]:
        break

print("Total disponible:", data["total"])
print("Total extraído:", len(all_products))

Total disponible: 194
Total extraído: 194


#### Ahora actualizamos los datos para que pasen de ser 30 incompletos a 194 en total

In [16]:
all_data = {
    "products": all_products,
    "total": len(all_products),
    "skip": 0,
    "limit": limit
}

In [17]:
file_path = bronze_path / "products_raw.json"

file_path.write_text(
    json.dumps(all_data, ensure_ascii=False, indent=4),
    encoding="utf-8"
)

503527

# Extraemos Clientes

In [21]:
url_users = "https://dummyjson.com/users"

response_users = requests.get(url_users, timeout = 30)

response_users.raise_for_status()

users_data = response_users.json()

print("Clientes disponibles: ", users_data["total"])
print("Clientes recibidos: ", len(users_data["users"]))

Clientes disponibles:  208
Clientes recibidos:  30


In [23]:
print(users_data.keys())
users_data["users"][0]

dict_keys(['users', 'total', 'skip', 'limit'])


{'id': 1,
 'firstName': 'Emily',
 'lastName': 'Johnson',
 'maidenName': 'Smith',
 'age': 29,
 'gender': 'female',
 'email': 'emily.johnson@x.dummyjson.com',
 'phone': '+81 965-431-3024',
 'username': 'emilys',
 'password': 'emilyspass',
 'birthDate': '1996-5-30',
 'image': 'https://dummyjson.com/icon/emilys/128',
 'bloodGroup': 'O-',
 'height': 193.24,
 'weight': 63.16,
 'eyeColor': 'Green',
 'hair': {'color': 'Brown', 'type': 'Curly'},
 'ip': '42.48.100.32',
 'address': {'address': '626 Main Street',
  'city': 'Phoenix',
  'state': 'Mississippi',
  'stateCode': 'MS',
  'postalCode': '29112',
  'coordinates': {'lat': -77.16213, 'lng': -92.084824},
  'country': 'United States'},
 'macAddress': '47:fa:41:18:ec:eb',
 'university': 'University of Wisconsin--Madison',
 'bank': {'cardExpire': '05/28',
  'cardNumber': '3693233511855044',
  'cardType': 'Diners Club International',
  'currency': 'GBP',
  'iban': 'GB74MH2UZLR9TRPHYNU8F8'},
 'company': {'department': 'Engineering',
  'name': 'Doo

In [25]:
users_df = pd.DataFrame(users_data["users"])

users_df.head(1)

,id,firstName,lastName,maidenName,age,gender,email,phone,username,password,...,address,macAddress,university,bank,company,ein,ssn,userAgent,crypto,role
0,1,Emily,Johnson,Smith,29,female,emily.johnson@x.dummyjson.com,+81 965-431-3024,emilys,emilyspass,...,"{'address': '626 Main Street', 'city': 'Phoeni...",47:fa:41:18:ec:eb,University of Wisconsin--Madison,"{'cardExpire': '05/28', 'cardNumber': '3693233...","{'department': 'Engineering', 'name': 'Dooley,...",977-175,900-590-289,Mozilla/5.0 (Macintosh; Intel Mac OS X 10_15_7...,"{'coin': 'Bitcoin', 'wallet': '0xb9fc2fe63b2a6...",admin


In [29]:
users_df.info()

<class 'pandas.DataFrame'>
RangeIndex: 30 entries, 0 to 29
Data columns (total 28 columns):
 #   Column      Non-Null Count  Dtype  
---  ------      --------------  -----  
 0   id          30 non-null     int64  
 1   firstName   30 non-null     str    
 2   lastName    30 non-null     str    
 3   maidenName  30 non-null     str    
 4   age         30 non-null     int64  
 5   gender      30 non-null     str    
 6   email       30 non-null     str    
 7   phone       30 non-null     str    
 8   username    30 non-null     str    
 9   password    30 non-null     str    
 10  birthDate   30 non-null     str    
 11  image       30 non-null     str    
 12  bloodGroup  30 non-null     str    
 13  height      30 non-null     float64
 14  weight      30 non-null     float64
 15  eyeColor    30 non-null     str    
 16  hair        30 non-null     object 
 17  ip          30 non-null     str    
 18  address     30 non-null     object 
 19  macAddress  30 non-null     str    
 20  u

In [30]:
print("Clientes disponibles:", users_data["total"])
print("Clientes recibidos:", len(users_data["users"]))

Clientes disponibles: 208
Clientes recibidos: 30


#### Extraccion de clientes automatizada

In [31]:
all_users = []

skip = 0
limit = 30

while True:
    response_users = requests.get(
        url_users,
        params={"limit": limit, "skip": skip},
        timeout=30
    )

    response_users.raise_for_status()

    batch = response_users.json()["users"]

    all_users.extend(batch)

    skip += limit

    if len(all_users) >= users_data["total"]:
        break

print("Total de clientes extraídos:", len(all_users))

Total de clientes extraídos: 208


#### los almacenamos

In [32]:
all_users_data = {
    "users": all_users,
    "total": len(all_users),
    "skip": 0,
    "limit": limit
}

In [33]:
users_file_path = bronze_path / "users_raw.json"

users_file_path.write_text(
    json.dumps(all_users_data, ensure_ascii=False, indent=4),
    encoding="utf-8"
)

530039

# Extraemos Carts

In [34]:
url_carts = "https://dummyjson.com/carts"

response_carts = requests.get(url_carts, timeout=30)

response_carts.raise_for_status()

carts_data = response_carts.json()

print("Carritos disponibles:", carts_data["total"])
print("Carritos recibidos:", len(carts_data["carts"]))

Carritos disponibles: 208
Carritos recibidos: 30


In [36]:
print(carts_data.keys())
carts_data["carts"][0]

dict_keys(['carts', 'total', 'skip', 'limit'])


{'id': 1,
 'products': [{'id': 162,
   'title': 'Blue Frock',
   'price': 29.99,
   'quantity': 4,
   'total': 119.96,
   'discountPercentage': 12.13,
   'discountedTotal': 105.41,
   'thumbnail': 'https://cdn.dummyjson.com/product-images/tops/blue-frock/thumbnail.webp'},
  {'id': 113,
   'title': 'Generic Motorcycle',
   'price': 3999.99,
   'quantity': 3,
   'total': 11999.97,
   'discountPercentage': 12.1,
   'discountedTotal': 10547.97,
   'thumbnail': 'https://cdn.dummyjson.com/product-images/motorcycle/generic-motorcycle/thumbnail.webp'},
  {'id': 122,
   'title': 'iPhone 6',
   'price': 299.99,
   'quantity': 3,
   'total': 899.97,
   'discountPercentage': 6.69,
   'discountedTotal': 839.76,
   'thumbnail': 'https://cdn.dummyjson.com/product-images/smartphones/iphone-6/thumbnail.webp'},
  {'id': 138,
   'title': 'Baseball Ball',
   'price': 8.99,
   'quantity': 2,
   'total': 17.98,
   'discountPercentage': 1.71,
   'discountedTotal': 17.67,
   'thumbnail': 'https://cdn.dummyjso

In [37]:
carts_df = pd.DataFrame(carts_data["carts"])

carts_df.head()

,id,products,total,discountedTotal,userId,totalProducts,totalQuantity
0,1,"[{'id': 162, 'title': 'Blue Frock', 'price': 2...",13037.88,11510.81,1,4,12
1,2,"[{'id': 86, 'title': 'Man Short Sleeve Shirt',...",139.93,125.70,2,2,7
2,3,"[{'id': 24, 'title': 'Fish Steak', 'price': 14...",1794.85,1625.77,3,6,15
3,4,"[{'id': 92, 'title': 'Sports Sneakers Off Whit...",689.93,636.81,4,2,7
4,5,"[{'id': 161, 'title': 'Samsung Galaxy Tab Whit...",1467.88,1205.80,5,3,12


In [38]:
carts_df.info()

<class 'pandas.DataFrame'>
RangeIndex: 30 entries, 0 to 29
Data columns (total 7 columns):
 #   Column           Non-Null Count  Dtype  
---  ------           --------------  -----  
 0   id               30 non-null     int64  
 1   products         30 non-null     object 
 2   total            30 non-null     float64
 3   discountedTotal  30 non-null     float64
 4   userId           30 non-null     int64  
 5   totalProducts    30 non-null     int64  
 6   totalQuantity    30 non-null     int64  
dtypes: float64(2), int64(4), object(1)
memory usage: 1.8+ KB


#### Automatizado

In [39]:
all_carts = []

skip = 0
limit = 30

while True:
    response_carts = requests.get(
        url_carts,
        params={"limit": limit, "skip": skip},
        timeout=30
    )

    response_carts.raise_for_status()

    batch = response_carts.json()["carts"]

    all_carts.extend(batch)

    skip += limit

    if len(all_carts) >= carts_data["total"]:
        break

print("Total de carritos extraídos:", len(all_carts))

Total de carritos extraídos: 208


In [40]:
all_carts_data = {
    "carts": all_carts,
    "total": len(all_carts),
    "skip": 0,
    "limit": limit
}

In [41]:
carts_file_path = bronze_path / "carts_raw.json"

carts_file_path.write_text(
    json.dumps(all_carts_data, ensure_ascii=False, indent=4),
    encoding="utf-8"
)

405257